In [1]:
import os
import re
import time
import uuid
import pickle
from pathlib import Path
from dotenv import load_dotenv

# .env 가 있는 폴더를 프로젝트 루트로 자동 탐색 (못 찾으면 현재 폴더)
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()),
    Path.cwd(),
)
print("프로젝트 루트 :", PROJECT_ROOT)

load_dotenv(PROJECT_ROOT / ".env")

try:
    openai_api_key = os.environ["OPENAI_API_KEY"]
    if not openai_api_key.strip():
        raise ValueError("OPENAI_API_KEY 값이 비어 있습니다. .env 파일을 확인해 주세요")
except KeyError:
    raise ValueError("OPENAI_API_KEY 키가 없습니다. .env 파일을 확인해 주세요")

processed = PROJECT_ROOT / "data" / "processed"

# 추출 결과 복원
with open(processed / "extracted.pkl", "rb") as f:
    doc1 = pickle.load(f)
    texts, tables, texts_2k_token = doc1["texts"], doc1["tables"], doc1["texts_2k_token"]

# 요약 결과 복원
with open(processed / "summaries.pkl", "rb") as f1:
    doc2 = pickle.load(f1)
    text_summaries = doc2["text_summaries"]
    table_summaries = doc2["table_summaries"]
    image_summaries = doc2["image_summaries"]
    img_base64_list = doc2["img_base64_list"]

print("복원 완료")
print(f"  텍스트 원본 {len(texts_2k_token):3d}건  / 요약 {len(text_summaries):3d}건")
print(f"  테이블 원본 {len(tables):3d}건  / 요약 {len(table_summaries):3d}건")
print(f"  이미지 원본 {len(img_base64_list):3d}건  / 요약 {len(image_summaries):3d}건")

프로젝트 루트 : c:\workspaces\RAG_Multimodal\multimodal-work
복원 완료
  텍스트 원본   6건  / 요약   6건
  테이블 원본   4건  / 요약   4건
  이미지 원본   6건  / 요약   6건


In [2]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 요약문이 들어갈 벡터 DB (= 색인 카드 서랍)
vectorstore = Chroma(
    collection_name="multimodal_rag",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)

# 원본이 들어갈 저장소 (= 서가)
store = InMemoryStore()

# 연결 키 (= 청구기호)
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore, docstore=store, id_key=id_key
)

print("검색기 생성 완료")

검색기 생성 완료


In [3]:
from langchain_core.documents import Document as _Doc


def add_documents(retriever, doc_summaries, doc_contents):
    """요약본은 벡터 DB에, 원본은 docstore에 '같은 id'로 저장합니다."""
    doc_ids = [str(uuid.uuid4()) for _ in doc_contents]

    summary_docs = [
        _Doc(page_content=s, metadata={id_key: doc_ids[i]})
        for i, s in enumerate(doc_summaries)
    ]

    # 요약본 → 벡터 DB (검색용)
    retriever.vectorstore.add_documents(summary_docs)

    # 원본 → docstore (전달용)
    retriever.docstore.mset(list(zip(doc_ids, doc_contents)))


if text_summaries:
    add_documents(retriever, text_summaries, texts_2k_token)
if table_summaries:
    add_documents(retriever, table_summaries, tables)
if image_summaries:
    add_documents(retriever, image_summaries, img_base64_list)

print("색인 완료")
print("  벡터 DB에 저장된 요약 수:", len(vectorstore.get()["ids"]))

색인 완료
  벡터 DB에 저장된 요약 수: 16


In [4]:
query = "[그림 1] 온라인 유통시장의 분기별 매출 규모 및 비중 추이 그래프에서 왼쪽 축과 오른쪽 축은 각각 무엇(단위 포함)을 나타내고, 그래프 맨 왼쪽 막대의 높이와 선의 위치는 대략 어느 정도이며, 맨 오른쪽까지 전체 추세는 어떻게 변했는지 설명해줘"

# 1. 질문을 임베딩으로 바꾼다  2. vectorstore 에서 가장 비슷한 요약을 찾는다(Chroma)
# 3. 찾은 요약의 metadata["doc_id"] 를 읽는다  4. docstore 에서 같은 doc_id 의 원본을 꺼낸다(InMemoryStore)
found = retriever.invoke(query)

print("검색된 원본 수:", len(found))
print()

for i, doc in enumerate(found):
    content = doc.page_content if hasattr(doc, "page_content") else doc
    is_image = bool(re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content))
    kind = "이미지(Base64)" if is_image else "텍스트/표"
    print(f"[{i}] {kind} — 길이 {len(content):,}자")
    print("   ", content[:120].replace("\n", " "), "...")
    print()

검색된 원본 수: 4

[0] 이미지(Base64) — 길이 185,392자
    /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/ ...

[1] 텍스트/표 — 길이 294자
    종속변수: 오프라인 매출 합계 (로그화) (1) 대형마트 (2) SSM (3) 할인점/슈퍼마켓 (4) 편의점 (5) 기타 전문유통업 1인당 온라인 지출 -0.264** 0.221*** -0.108 0.324*** 0 ...

[2] 텍스트/표 — 길이 2,912자
    0  2016년 2017년 2018년 2019년 2020년 2021년 2022년 2023년 2024년  주: 막대그래프는 온라인 유통시장 매출 규모(좌축, 조원)를, 선 그래프는 온라인 유통매출 비중(우축, %)을  ...

[3] 텍스트/표 — 길이 144자
    우선 오프라인 유통업체들의 매출을 요인별로 구분하여 그 기여도를 분석했다. <표 1> 결제 1건당 매출(2)이나 소비자 1인당 결제 건수(3)와 같은 질적인 측면에서는 소비자 들의 온라인 지출이 증가함에도 불구하고  ...



> ✅ 검색 결과에 **이미지(Base64)가 최소 1건** 들어 있어야 합니다.

In [5]:
from langchain_core.documents import Document


def split_image_text_types(docs):
    """검색 결과를 Base64 이미지와 텍스트로 나눕니다."""
    b64_images = []
    texts_out = []

    for doc in docs:
        content = doc.page_content if isinstance(doc, Document) else doc
        # Base64 문자열은 영문/숫자/+/= 만으로 길게 이어집니다
        if re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content):
            b64_images.append(content)
        else:
            texts_out.append(content)

    return {"images": b64_images, "texts": texts_out}


split_data = split_image_text_types(found)
print("이미지:", len(split_data["images"]), "건")
print("텍스트/표:", len(split_data["texts"]), "건")

이미지: 1 건
텍스트/표: 3 건


In [6]:
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI


def img_prompt_func(data_dict):
    """검색된 원본을 멀티모달 LLM이 읽을 형태로 조립합니다."""
    messages = []

    # 1) 이미지를 먼저 넣습니다 (모델이 그림을 먼저 보게)
    for image in data_dict["context"]["images"]:
        messages.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image}"},
        })

    # 2) 텍스트 컨텍스트와 질문을 넣습니다
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages.append({
        "type": "text",
        "text": (
            "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
            "자료에 없는 내용은 모른다고 답하세요.\n\n"
            f"질문: {data_dict['question']}\n\n"
            f"참고 텍스트:\n{formatted_texts}"
        ),
    })

    return [HumanMessage(content=messages)]


model = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)

chain = (
    {
        "context": retriever | RunnableLambda(split_image_text_types),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(img_prompt_func)
    | model
    | StrOutputParser()
)

print("최종 체인 조립 완료")

최종 체인 조립 완료


In [7]:
start = time.time()
answer = chain.invoke(query)

print(f"소요 시간: {time.time() - start:.2f}초")
print(f"질문(Q): {query}")
print("-" * 80)
print(f"답변(A): {answer}")

소요 시간: 6.53초
질문(Q): [그림 1] 온라인 유통시장의 분기별 매출 규모 및 비중 추이 그래프에서 왼쪽 축과 오른쪽 축은 각각 무엇(단위 포함)을 나타내고, 그래프 맨 왼쪽 막대의 높이와 선의 위치는 대략 어느 정도이며, 맨 오른쪽까지 전체 추세는 어떻게 변했는지 설명해줘
--------------------------------------------------------------------------------
답변(A): [그림 1]에서 왼쪽 축은 온라인 유통시장 매출 규모를 나타내며, 단위는 조원입니다. 오른쪽 축은 온라인 유통매출 비중을 나타내며, 단위는 퍼센트(%)입니다.

그래프 맨 왼쪽 막대의 높이는 대략 10조원 정도이며, 선의 위치는 약 40% 정도입니다. 전체 추세를 보면, 온라인 유통시장의 매출 규모는 시간이 지남에 따라 지속적으로 증가하고 있으며, 비중 또한 상승하는 경향을 보이고 있습니다. 특히, 최근 몇 년간 매출과 비중 모두 상승세를 유지하고 있는 것으로 나타납니다.


In [8]:
# 8-1. 비교용 텍스트 전용 RAG (이미지·표 제외, 텍스트 청크만)
vectorstore_text_only = Chroma(
    collection_name="text_only_rag",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)
vectorstore_text_only.add_texts(texts_2k_token)

prompt_text_only = ChatPromptTemplate.from_template(
    "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
    "자료에 없는 내용은 모른다고 답하세요.\n\n"
    "질문: {question}\n\n"
    "참고 텍스트:\n{context}"
)

chain_text_only = (
    {
        "context": vectorstore_text_only.as_retriever(search_kwargs={"k": 4})
        | RunnableLambda(lambda docs: "\n".join(d.page_content for d in docs)),
        "question": RunnablePassthrough(),
    }
    | prompt_text_only
    | model
    | StrOutputParser()
)

print("비교용 텍스트 전용 RAG 준비 완료")

비교용 텍스트 전용 RAG 준비 완료


In [9]:
# 8-2. Before / After 출력
print("─" * 50)
print(f"질문: {query}")
print()

before = chain_text_only.invoke(query)
print(f"[BEFORE | 텍스트 전용 RAG]\n{before}")
print()

after = chain.invoke(query)
print(f"[AFTER | 멀티모달 RAG]\n{after}")

──────────────────────────────────────────────────
질문: [그림 1] 온라인 유통시장의 분기별 매출 규모 및 비중 추이 그래프에서 왼쪽 축과 오른쪽 축은 각각 무엇(단위 포함)을 나타내고, 그래프 맨 왼쪽 막대의 높이와 선의 위치는 대략 어느 정도이며, 맨 오른쪽까지 전체 추세는 어떻게 변했는지 설명해줘

[BEFORE | 텍스트 전용 RAG]
[그림 1] 온라인 유통시장의 분기별 매출 규모 및 비중 추이 그래프에서 왼쪽 축은 온라인 유통시장 매출 규모(단위: 조원)를 나타내고, 오른쪽 축은 온라인 유통매출 비중(단위: %)을 나타냅니다. 그래프 맨 왼쪽 막대의 높이는 대략 10조원 정도이며, 선의 위치는 약 20% 정도입니다. 맨 오른쪽까지 전체 추세는 온라인 유통시장의 매출 규모가 지속적으로 증가하고 있으며, 비중 또한 증가하여 2023년에는 50%를 돌파하고 2026년 3월에는 60%에 이르는 추세를 보이고 있습니다.

[AFTER | 멀티모달 RAG]
[그림 1]에서 왼쪽 축은 온라인 유통시장 매출 규모를 나타내며, 단위는 조원입니다. 오른쪽 축은 온라인 유통매출 비중을 나타내며, 단위는 퍼센트(%)입니다.

그래프 맨 왼쪽 막대의 높이는 대략 10조원 정도이며, 선의 위치는 약 40% 정도로 보입니다. 맨 오른쪽까지 전체 추세는 온라인 유통시장의 매출 규모가 지속적으로 증가하고 있으며, 비중 또한 상승하는 경향을 보이고 있습니다. 전체적으로 온라인 유통시장이 성장하고 있음을 나타냅니다.


In [10]:
query_table = "<표 1> 온라인 거래 확대가 오프라인 시장에 미치는 효과(기여도 분석)에서 관측치 수와 읍면동 수는 각각 얼마야?"

print("─" * 50)
print(f"질문: {query_table}")
print()

before_table = chain_text_only.invoke(query_table)
print(f"[BEFORE | 텍스트 전용 RAG]\n{before_table}")
print()

after_table = chain.invoke(query_table)
print(f"[AFTER | 멀티모달 RAG]\n{after_table}")

──────────────────────────────────────────────────
질문: <표 1> 온라인 거래 확대가 오프라인 시장에 미치는 효과(기여도 분석)에서 관측치 수와 읍면동 수는 각각 얼마야?

[BEFORE | 텍스트 전용 RAG]
자료에 따르면, <표 1> 온라인 거래 확대가 오프라인 시장에 미치는 효과(기여도 분석)에서 관측치 수와 읍면동 수에 대한 구체적인 숫자는 제공되지 않았습니다. 따라서 이와 관련된 정보는 알 수 없습니다.

[AFTER | 멀티모달 RAG]
<표 1> 온라인 거래 확대가 오프라인 시장에 미치는 효과(기여도 분석)에서 관측치 수는 197,089이며, 읍면동 수는 3,518입니다.


In [11]:
questions = [
    "[그림 1] 온라인 유통시장의 분기별 매출 규모 및 비중 추이 그래프에서 왼쪽 축과 오른쪽 축은 각각 무엇(단위 포함)을 나타내고, 그래프 맨 왼쪽 막대의 높이와 선의 위치는 대략 어느 정도이며, 맨 오른쪽까지 전체 추세는 어떻게 변했는지 설명해줘",
    "<표 2> 업태별 분석에서 '(1) 대형마트' 열의 1인당 온라인 지출 계수와 표준오차, 시군구 수는 얼마야?",
    "<표 1> 기여도 분석에서 종속변수가 '업체 수'인 모형의 1인당 온라인 지출 계수와 유의수준, 관측치 수와 읍면동 수는 얼마야?",
    "<표 3> 로켓배송 도입 효과에서 '로켓배송 도입' 행의 (1) 매출 열 계수와 표준오차, 별(*) 개수는 얼마야?",
    "이 보고서가 제시한 유통정책의 시사점 네 가지는 무엇이야?",
]

table_set = set(tables)


def classify(doc):
    """검색된 원본이 이미지 / 표 / 텍스트 중 무엇인지 구분합니다."""
    content = doc.page_content if hasattr(doc, "page_content") else doc
    if re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content):
        return "이미지"
    if content in table_set:
        return "표"
    return "텍스트"


rows = []
for q in questions:
    docs = retriever.invoke(q)
    kinds = [classify(d) for d in docs]
    summary_kinds = " + ".join(f"{k} {kinds.count(k)}" for k in ["이미지", "표", "텍스트"] if kinds.count(k))
    ans = chain.invoke(q)
    rows.append((q, ans, summary_kinds))
    time.sleep(1)  # RateLimitError 방지

print("| # | 질문 | 답변 | 근거가 된 원본 종류 |")
print("| --- | --- | --- | --- |")
for i, (q, a, k) in enumerate(rows, 1):
    q1 = q.replace("|", "/").replace("\n", " ")
    a1 = a.replace("|", "/").replace("\n", " ")
    print(f"| {i} | {q1} | {a1} | {k} |")

| # | 질문 | 답변 | 근거가 된 원본 종류 |
| --- | --- | --- | --- |
| 1 | [그림 1] 온라인 유통시장의 분기별 매출 규모 및 비중 추이 그래프에서 왼쪽 축과 오른쪽 축은 각각 무엇(단위 포함)을 나타내고, 그래프 맨 왼쪽 막대의 높이와 선의 위치는 대략 어느 정도이며, 맨 오른쪽까지 전체 추세는 어떻게 변했는지 설명해줘 | [그림 1]에서 왼쪽 축은 온라인 유통시장 매출 규모를 나타내며, 단위는 조원입니다. 오른쪽 축은 온라인 유통매출 비중을 나타내며, 단위는 퍼센트(%)입니다.  그래프 맨 왼쪽 막대의 높이는 대략 10조원 정도이며, 선의 위치는 약 40% 정도로 보입니다. 맨 오른쪽까지 전체 추세는 온라인 유통시장의 매출 규모가 지속적으로 증가하고 있으며, 비중 또한 상승하는 경향을 보이고 있습니다. 전체적으로 온라인 유통시장이 성장하고 있음을 나타냅니다. | 이미지 1 + 표 2 + 텍스트 1 |
| 2 | <표 2> 업태별 분석에서 '(1) 대형마트' 열의 1인당 온라인 지출 계수와 표준오차, 시군구 수는 얼마야? | <표 2> 업태별 분석에서 '(1) 대형마트' 열의 1인당 온라인 지출 계수는 -0.264이며, 표준오차는 0.113입니다. 시군구 수는 173입니다. | 표 3 + 텍스트 1 |
| 3 | <표 1> 기여도 분석에서 종속변수가 '업체 수'인 모형의 1인당 온라인 지출 계수와 유의수준, 관측치 수와 읍면동 수는 얼마야? | <표 1> 기여도 분석에서 종속변수가 '업체 수'인 모형의 1인당 온라인 지출 계수는 0.152***이며, 유의수준은 ***입니다. 관측치 수는 197,089이고, 읍면동 수는 3,518입니다. | 표 4 |
| 4 | <표 3> 로켓배송 도입 효과에서 '로켓배송 도입' 행의 (1) 매출 열 계수와 표준오차, 별(*) 개수는 얼마야? | '로켓배송 도입' 행의 (1) 매출 열 계수는 0.053, 표준오차는 0.030이며, 별(*) 개수는 1개입니다. | 표 3 + 텍스트 1 |
| 